In [1]:
import numpy as np
# for auto-reloading modules
%load_ext autoreload
%autoreload 2

In [2]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from project_utils import *
from project_utils.dataset_utils import *
from project_utils.MTDMD import mtdmd

# Open plt figs in separate interactive windows: https://stackoverflow.com/a/14277370
%matplotlib qt

# DMD-based Control
**Notebook Tasks**
1. Collect $X_{train}$, find its shape
2. Calculate the DMD, find how many PC Modes correspond to 70%, 80%, 90%, and 95% energy with a *cumulative energy plot*
3. For each of the 95% energy DMD modes, *plot the transitions 4D*
4. Calculate the covariance matrix and threshold the correlation matrix to get dynamics $A$ in latent space
5. Find the minimum number of nodes needed to control the system

## 1. Collect $X_{train}$, find its shape, *plot as 4D animations*

In [3]:
train_folder = './data/train/'
test_folder = './data/test/'

JUMPING, RUNNING, WALKING = 0, 1, 2

COLORS = {JUMPING:(1,0,0,1), RUNNING:(0,1,0,1), WALKING:(0,0,1,1)}

train_fnames = [
    'jumping_1', 'jumping_2', 'jumping_3', 'jumping_4', 'jumping_5',
    'running_1', 'running_2', 'running_3', 'running_4', 'running_5',
    'walking_1', 'walking_2', 'walking_3', 'walking_4', 'walking_5'
]

train_labels = [
    JUMPING, JUMPING, JUMPING, JUMPING, JUMPING,
    RUNNING, RUNNING, RUNNING, RUNNING, RUNNING,
    WALKING, WALKING, WALKING, WALKING, WALKING
]

if REMOVE_OUTLIERS := False:
    train_fnames = [
        'jumping_1', 'jumping_2', 'jumping_3', 'jumping_4', 'jumping_5',
        'running_1', 'running_3', 'running_4', 'running_5',
        'walking_2', 'walking_3', 'walking_4', 'walking_5'
    ]
    train_labels = [
        JUMPING, JUMPING, JUMPING, JUMPING, JUMPING,
        RUNNING, RUNNING, RUNNING, RUNNING,
        WALKING, WALKING, WALKING, WALKING
    ]

num_jumping = sum(1 if l == JUMPING else 0 for l in train_labels)
num_running = sum(1 if l == RUNNING else 0 for l in train_labels)
num_walking = sum(1 if l == WALKING else 0 for l in train_labels)

run_idx = num_jumping
walk_idx = num_jumping+num_running

test_fnames = ['jumping_1t', 'running_1t', 'walking_1t']

test_labels = [JUMPING, RUNNING, WALKING]

In [4]:
train_data_tensor, test_data_tensor = get_train_test_set(train_folder, train_fnames, test_folder, test_fnames, as_matrix=False)
print('train data shape', train_data_tensor.shape, '|', 'test data shape', test_data_tensor.shape)

train data shape (15, 114, 100) | test data shape (3, 114, 100)


### 2. Calculate the DMD, find how many PC Modes correspond to 70%, 80%, 90%, and 95% energy with a *cumulative energy plot*

In [9]:
dmd_fit = mtdmd(train_data_tensor.transpose((0, 2, 1)), 0.97)

[MTDMD] Chosen rank r = 3  (explains 97.4% of variance, threshold = 97.0%)


In [6]:
print('Effective rank:', int(sum(dmd_fit['singular_values'] > 1e-6)))
print('First 10 Singular values:', [float(round(sig, 2)) for sig in dmd_fit['singular_values'][:10]])

Effective rank: 114
First 10 Singular values: [16199467694.92, 877868308.62, 418291168.66, 181918112.4, 87683535.82, 58243340.86, 37812912.54, 29187606.9, 17012882.61, 15177045.42]


In [10]:
energy = dmd_fit['singular_values']**2/np.sum(dmd_fit['singular_values']**2)
cumsum_energy = np.cumsum(energy)

idx_80 = np.argmax(cumsum_energy >= 0.8)
idx_90 = np.argmax(cumsum_energy >= 0.9)
idx_95 = np.argmax(cumsum_energy >= 0.95)
idx_97 = np.argmax(cumsum_energy >= 0.97)

plt.plot(cumsum_energy, c='k')

plt.hlines(0.8, xmin=0, xmax=energy.shape[0]-1, colors='tab:orange', linestyles=':', label=f'80% Var explained by {idx_80+1} DMD Modes')
plt.hlines(0.9, xmin=0, xmax=energy.shape[0]-1, colors='tab:green', linestyles=':', label=f'90% Var explained by {idx_90+1} DMD Modes')
plt.hlines(0.95, xmin=0, xmax=energy.shape[0]-1, colors='tab:red', linestyles=':', label=f'95% Var explained by {idx_95+1} DMD Modes')
plt.hlines(0.97, xmin=0, xmax=energy.shape[0]-1, colors='tab:red', linestyles=':', label=f'97% Var explained by {idx_97+1} DMD Modes')
plt.xlabel('DMD Mode index $j$')
plt.ylabel('$E_j$')
plt.title(f'Energy vs Singular Value Index')
plt.legend()

plt.savefig('./figs/DMD_Energy.png')
plt.close()

In [16]:
# plot first 5 DMD modes as animations
if SAVE_R_DMD_ANIMS := True:
  for i in range(dmd_fit['rank']):
    plot_action(
	    np.vstack(tuple(train_data_tensor[:,:,0].mean(0)*(dmd_fit['modes'][:, i]**(p+1)).real for p in range(20))),
	    f'DMD_Mode_{i+1}'
    )

In [12]:
# plot average activity projections
reconstructed_jumping = dmd_fit['reconstructions'][:run_idx, :, :].mean(axis=0)
reconstructed_running = dmd_fit['reconstructions'][run_idx:walk_idx, :, :].mean(axis=0)
reconstructed_walking = dmd_fit['reconstructions'][walk_idx:, :, :].mean(axis=0)

if SAVE_MEAN_ACT_ANIMS := False:
  plot_action(reconstructed_jumping.T, 'DMD_Low_Dim_Jumping')
  plot_action(reconstructed_running.T, 'DMD_Low_Dim_Running')
  plot_action(reconstructed_walking.T, 'DMD_Low_Dim_Walking')

## 3. *Plot train samples in 2D and 3D PC space*

In [13]:
fig = plt.figure()
ax = fig.add_subplot()

pca2 = PCA(2)
X_train_2Dproj = pca2.fit_transform(train_data_tensor).T

# plot the data vector tails
ax.scatter(X_train_2Dproj[0, :], X_train_2Dproj[1, :], c=[COLORS[c] for c in train_labels])
# plot the data vector shafts
for i in range(X_train_2Dproj.shape[1]):
    ax.plot([0, X_train_2Dproj[0, i]], [0, X_train_2Dproj[1, i]], c=COLORS[train_labels[i]])
# plot the jumping, running, and walking centroids
ax.scatter([X_train_2Dproj[0, :run_idx].mean()], [X_train_2Dproj[1, :run_idx].mean()], color=COLORS[0], marker='x')
ax.scatter([X_train_2Dproj[0, run_idx:walk_idx].mean()], [X_train_2Dproj[1, run_idx:walk_idx].mean()], color=COLORS[1], marker='x')
ax.scatter([X_train_2Dproj[0, walk_idx:].mean()], [X_train_2Dproj[1, walk_idx:].mean()], color=COLORS[2], marker='x')

max_ax_val = np.max(np.abs(X_train_2Dproj))
ax.set_xlim(-max_ax_val, max_ax_val)
ax.set_xlabel('PC 1')
ax.set_ylim(-max_ax_val, max_ax_val)
ax.set_ylabel('PC 2')
# Custom legend: https://stackoverflow.com/a/39500357
run_p, jump_p, walk_p = mpatches.Patch(color=COLORS[JUMPING], label='Jumping'), mpatches.Patch(color=COLORS[RUNNING], label='Running'), mpatches.Patch(color=COLORS[WALKING], label='Walking')
plt.legend(handles=[run_p, jump_p, walk_p])
plt.title('Projection of Train Data onto Top 2 POD Modes')
plt.show()

del pca2

In [14]:
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

pca3 = PCA(3)
X_train_3Dproj = pca3.fit_transform(train_data_tensor).T

# 3D matplotlib: https://matplotlib.org/stable/gallery/mplot3d/2dcollections3d.html#sphx-glr-gallery-mplot3d-2dcollections3d-py
# plot the data vector tails
ax.scatter(X_train_3Dproj[0, :], X_train_3Dproj[1, :], zs=X_train_3Dproj[2, :], c=[COLORS[c] for c in train_labels])
# plot the data vector shafts
for i in range(X_train_3Dproj.shape[1]):
    ax.plot([0, X_train_3Dproj[0, i]], [0, X_train_3Dproj[1, i]], zs=[0, X_train_3Dproj[2, i]], zdir='z', c=COLORS[train_labels[i]])
# plot the jumping, running, and walking centroids
ax.scatter([X_train_3Dproj[0, :run_idx].mean()], [X_train_3Dproj[1, :run_idx].mean()], [X_train_3Dproj[2, :run_idx].mean()], color=COLORS[0], marker='x')
ax.scatter([X_train_3Dproj[0, run_idx:walk_idx].mean()], [X_train_3Dproj[1, run_idx:walk_idx].mean()], [X_train_3Dproj[2, run_idx:walk_idx].mean()], color=COLORS[1], marker='x')
ax.scatter([X_train_3Dproj[0, walk_idx:].mean()], [X_train_3Dproj[1, walk_idx:].mean()], [X_train_3Dproj[2, walk_idx:].mean()], color=COLORS[2], marker='x')

max_ax_val = np.max(np.abs(X_train_3Dproj))
ax.set_xlim3d(-max_ax_val, max_ax_val)
ax.set_xlabel('PC 1')
ax.set_ylim3d(-max_ax_val, max_ax_val)
ax.set_ylabel('PC 2')
ax.set_zlim3d(-max_ax_val, max_ax_val)
ax.set_zlabel('PC 3')
# Custom legend: https://stackoverflow.com/a/39500357
run_p, jump_p, walk_p = mpatches.Patch(color=(1,0,0,1), label='Jumping'), mpatches.Patch(color=(0,1,0,1), label='Running'), mpatches.Patch(color=(0,0,1,1), label='Walking')
plt.legend(handles=[run_p, jump_p, walk_p])
plt.title('Projection of Train Data onto Top 3 PC Modes')
plt.show()

del pca3

## 4. Find class centroids in $k$-PC mode space

In [15]:
k = idx_95+1

train_kD_proj = np.einsum('ND,MD->MN', train_data_tensor, pca.components_[:k, :])

fig = plt.figure()
ax = fig.add_subplot()
cax = ax.matshow(train_kD_proj, interpolation='nearest')
fig.colorbar(cax, shrink=0.5)

labels = ['']*train_kD_proj.shape[1]
labels[2], labels[7], labels[12] = 'jumping', 'running', 'walking'
ax.tick_params(top=False, labeltop=True, bottom=True, labelbottom=False)
ax.set_xticks(np.arange(train_kD_proj.shape[1]), labels=labels)
ax.set_ylabel(f'Coordinate in {k}D POD Space')
plt.title(f'Train Samples Projected onto k={k} POD Modes\n')
plt.show();

In [16]:
jump_centroid = np.mean(train_kD_proj[:, :run_idx], axis=1)
run_centroid = np.mean(train_kD_proj[:, run_idx:walk_idx], axis=1)
walk_centroid = np.mean(train_kD_proj[:, walk_idx:], axis=1)

centroids = np.stack((jump_centroid, run_centroid, walk_centroid))

fig = plt.figure()
ax = fig.add_subplot()
cax = ax.matshow(centroids.T, interpolation='nearest')
fig.colorbar(cax, shrink=0.8)

CLASS_NAMES = ['jumping', 'running', 'walking']
ax.set_xticks(np.arange(3), CLASS_NAMES)
ax.set_ylabel(f'Coordinate in {k}D POD Space')
plt.title(f'Train Class Centroids in {k}D POD Space\n')
plt.show();